# 03 — Build modeling splits (train/val pool + holdout)

**Summary:** Load labeled training data (ranked, clean, or raw), clean prompts/SVGs, optionally merge **difficulty metadata** from `train_ranked.csv` when needed, optionally keep only the **easiest `EASY_SUBSET_FRAC`** of rows (`USE_EASY_SUBSET`), then optionally the **first `FIRST_N_LABELED` rows**, then reserve **`HOLDOUT_N` rows** for holdout. Saves under `outputs/workflow_runs/<RUN_PROFILE_ID>/modeling_splits/` plus `workflow_layout.json` at the profile root.

**Prerequisites:** Notebooks **01** and **02** if you use ranked data or easy subset without difficulty columns in the chosen file; otherwise raw `train.csv` is enough.

**Profiles:** Set **`RUN_PROFILE_ID`** (filesystem-safe slug) when you change `SEED`, `HOLDOUT_N`, `FIRST_N_LABELED`, or easy-subset settings so artifacts do not collide. Use `RUN_PROFILE_ID="default"` for a single canonical tree.

**Stable holdout:** With `REUSE_EXISTING_SPLITS = True` (default), reruns load existing CSVs when the manifest matches parameters. Set `FORCE_REBUILD_SPLITS = True` to overwrite.

**Next:** Run notebook **04** using the same `RUN_PROFILE_ID` / `WORKFLOW_ROOT` as here.

#### Colab: mount Google Drive (optional)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#### Install dependencies

In [2]:
!pip -q install transformers peft accelerate bitsandbytes datasets trl cairosvg pillow scikit-learn lxml pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.7 MB/s eta 0:00:00


#### Imports, paths

In [3]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path('/content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL')
# For local runs, uncomment and set:
# PROJECT_DIR = Path(__file__).resolve().parents[1]

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

RAW_DIR = PROJECT_DIR / 'data' / 'raw'
INTERIM_DIR = PROJECT_DIR / 'data' / 'interim'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
OUTPUTS_DIR = PROJECT_DIR / 'outputs'

print('PROJECT_DIR:', PROJECT_DIR)

PROJECT_DIR: /content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL


#### Parameters

In [4]:
SEED = 42
HOLDOUT_N = 100 #100
# After cleaning: use only the first N rows in table order (None = all rows).
FIRST_N_LABELED = None  # e.g. 2000

# Isolate all downstream artifacts (splits, models, tuning CSVs, evals) under this profile.
RUN_PROFILE_ID = 'run_1'  # change when SEED / HOLDOUT_N / FIRST_N / easy subset / data source changes
WORKFLOW_ROOT = PROJECT_DIR / 'outputs' / 'workflow_runs' / RUN_PROFILE_ID

# Easiest fraction of difficulty-ranked rows (after optional merge from train_ranked.csv).
USE_EASY_SUBSET = False
EASY_SUBSET_FRAC = 0.2

REUSE_EXISTING_SPLITS = True
FORCE_REBUILD_SPLITS = False

print(
    'WORKFLOW_ROOT:', WORKFLOW_ROOT,
    '| HOLDOUT_N:', HOLDOUT_N,
    '| FIRST_N_LABELED:', FIRST_N_LABELED,
    '| USE_EASY_SUBSET:', USE_EASY_SUBSET,
    '| REUSE_EXISTING_SPLITS:', REUSE_EXISTING_SPLITS,
    '| FORCE_REBUILD_SPLITS:', FORCE_REBUILD_SPLITS,
)

WORKFLOW_ROOT: /content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL/outputs/workflow_runs/run_1 | HOLDOUT_N: 100 | FIRST_N_LABELED: None | USE_EASY_SUBSET: False | REUSE_EXISTING_SPLITS: True | FORCE_REBUILD_SPLITS: False


#### Load data, clean, build holdout + pool, save splits

In [5]:
from src.core.modeling_splits import (
    build_pool_and_holdout,
    load_existing_split_tables,
    load_split_manifest,
    manifest_matches_params,
    split_artifacts_exist,
)
from src.core.workflow_layout import write_workflow_layout_stub

RANKED_PATH = PROCESSED_DIR / 'train_ranked.csv'
CLEAN_PATH = INTERIM_DIR / 'train_clean_basic.csv'
RAW_TRAIN_PATH = RAW_DIR / 'train.csv'

if RANKED_PATH.exists():
    data_path = RANKED_PATH
elif CLEAN_PATH.exists():
    data_path = CLEAN_PATH
else:
    data_path = RAW_TRAIN_PATH


if FORCE_REBUILD_SPLITS or not REUSE_EXISTING_SPLITS or not split_artifacts_exist(WORKFLOW_ROOT):
    train_val_pool, holdout_df = build_pool_and_holdout(
        data_path=data_path,
        workflow_root=WORKFLOW_ROOT,
        seed=SEED,
        holdout_n=HOLDOUT_N,
        run_profile_id=str(RUN_PROFILE_ID),
        use_easy_subset=bool(USE_EASY_SUBSET),
        easy_subset_frac=float(EASY_SUBSET_FRAC),
        first_n_labeled=FIRST_N_LABELED,
        ranked_path=RANKED_PATH,
    )
else:
    manifest = load_split_manifest(WORKFLOW_ROOT)
    ok, msg = manifest_matches_params(
        manifest,
        seed=SEED,
        holdout_n=HOLDOUT_N,
        source_csv=str(data_path),
        first_n_labeled=FIRST_N_LABELED,
        run_profile_id=str(RUN_PROFILE_ID),
        use_easy_subset=bool(USE_EASY_SUBSET),
        easy_subset_frac=float(EASY_SUBSET_FRAC) if USE_EASY_SUBSET else None,
    )
    if not ok:
        raise ValueError(
            f'Cannot reuse splits: {msg}. Set FORCE_REBUILD_SPLITS = True to regenerate, '
            'or align SEED / HOLDOUT_N / FIRST_N_LABELED / RUN_PROFILE_ID / easy subset / data source.'
        )
    train_val_pool, holdout_df = load_existing_split_tables(WORKFLOW_ROOT)
    print('Reused splits from disk (holdout unchanged).', msg)

_man = load_split_manifest(WORKFLOW_ROOT)
write_workflow_layout_stub(
    PROJECT_DIR,
    WORKFLOW_ROOT,
    run_profile_id=str(RUN_PROFILE_ID),
    split_manifest=_man,
)

print('Pool rows:', len(train_val_pool), '| Holdout:', len(holdout_df))

Loaded: /content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL/data/processed/train_ranked.csv (50000, 64)
Saved: /content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL/outputs/workflow_runs/run_1/modeling_splits/train_val_pool.csv
Saved: /content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL/outputs/workflow_runs/run_1/modeling_splits/holdout_eval.csv
Pool rows: 49900 | Holdout: 100
